# UrbanPulse Nepal — BIPAD Incident Data Acquisition

This first notebook retrieves historical incident records from the public BIPAD Portal API, follows pagination links returned by the API, preserves raw page responses, and reports what the retrieval actually covered.

Run top to bottom. It makes live requests and writes a new timestamped folder under `data/raw/bipad_incidents/`. It does not overwrite prior runs. An API-reported total does not by itself establish complete historical coverage.

Source: [BIPAD API documentation](https://bipadportal.gov.np/api/) — `/api/v1/incident/`.

## 1. Configure the retrieval

The notebook follows API-provided `next` links instead of assuming a page parameter. Requests use a timeout, retries, and a maximum-page guard.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse
import hashlib, json, time
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').is_dir(): PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'data').is_dir(): raise FileNotFoundError('Run this notebook from the repository or notebooks/ folder.')
API_BASE = 'https://bipadportal.gov.np/api/v1'
INCIDENT_ENDPOINT = f'{API_BASE}/incident/'
OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'bipad_incidents'
REQUEST_TIMEOUT_SECONDS = 45
MAX_PAGES = 500  # Safety guard; increase only after reviewing the observed API volume.
PAGE_DELAY_SECONDS = 0.15
retrieved_at = datetime.now(timezone.utc)
run_id = retrieved_at.strftime('%Y%m%dT%H%M%SZ')
run_dir = OUTPUT_ROOT / run_id
run_dir.mkdir(parents=True, exist_ok=False)
session = requests.Session()
retry = Retry(total=5, connect=5, read=5, status=5, backoff_factor=1, status_forcelist=(429,500,502,503,504), allowed_methods=frozenset({'GET'}), respect_retry_after_header=True)
session.mount('https://', HTTPAdapter(max_retries=retry))
session.headers.update({'Accept':'application/json', 'User-Agent':'UrbanPulseNepal-research/0.1'})
print(f'Project root: {PROJECT_ROOT}')
print(f'Endpoint: {INCIDENT_ENDPOINT}')
print(f'Run output: {run_dir}')

Project root: c:\Projects_Tracking\DisasterLens Nepal
Endpoint: https://bipadportal.gov.np/api/v1/incident/
Run output: c:\Projects_Tracking\DisasterLens Nepal\data\raw\bipad_incidents\20261013T101245Z


## 2. Retrieve and preserve API responses

Every response page is saved before continuing. The loop supports a paginated `results` object or a plain JSON list, follows same-host `next` links, and records incomplete or unusual responses rather than claiming a complete download.

In [2]:
pages, records, page_errors, visited_urls = [], [], [], set()
reported_total = None
next_url = INCIDENT_ENDPOINT
stop_reason = 'unknown'
for page_number in range(1, MAX_PAGES + 1):
    if not next_url:
        stop_reason = 'API pagination ended (no next page).'
        break
    if next_url in visited_urls:
        stop_reason = 'Repeated pagination URL detected; stopped to prevent a loop.'
        page_errors.append({'page':page_number, 'error':'Repeated pagination URL'})
        break
    if urlparse(next_url).netloc.lower() != urlparse(INCIDENT_ENDPOINT).netloc.lower():
        stop_reason = 'Pagination link points outside BIPAD host; stopped for safety.'
        page_errors.append({'page':page_number, 'error':'External pagination URL rejected'})
        break
    visited_urls.add(next_url)
    try:
        response = session.get(next_url, timeout=REQUEST_TIMEOUT_SECONDS)
        response.raise_for_status()
        payload = response.json()
    except (requests.RequestException, ValueError) as exc:
        stop_reason = f'Request or JSON parsing failed on page {page_number}.'
        page_errors.append({'page':page_number, 'url':next_url, 'error':str(exc)})
        break
    raw_path = run_dir / f'page_{page_number:05d}.json'
    raw_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    pages.append({'page':page_number, 'url':next_url, 'status_code':response.status_code, 'file':raw_path.name})
    if isinstance(payload, dict) and isinstance(payload.get('results'), list):
        page_records = payload['results']
        if reported_total is None: reported_total = payload.get('count')
        candidate_next = payload.get('next')
    elif isinstance(payload, list):
        page_records, candidate_next = payload, None
    else:
        stop_reason = f'Unexpected API response structure on page {page_number}.'
        page_errors.append({'page':page_number, 'error':f'Unexpected payload type: {type(payload).__name__}'})
        break
    if not page_records:
        stop_reason = f'Empty results on page {page_number}.'
        break
    records.extend(page_records)
    if not candidate_next:
        stop_reason = 'API pagination ended (no next page).'
        break
    next_url = requests.compat.urljoin(response.url, candidate_next)
    time.sleep(PAGE_DELAY_SECONDS)
else:
    stop_reason = f'MAX_PAGES safety guard reached ({MAX_PAGES}); further pages may remain.'
retrieval_complete = stop_reason == 'API pagination ended (no next page).' and not page_errors
print(f'Pages saved: {len(pages)}')
print(f'Record entries received: {len(records)}')
print(f'API-reported total (if provided): {reported_total}')
print(f'Pagination complete: {retrieval_complete}')
print(f'Stopping reason: {stop_reason}')
if page_errors: display(pd.DataFrame(page_errors))

Pages saved: 66
Record entries received: 64735
API-reported total (if provided): 9223372036854775807
Pagination complete: False
Stopping reason: Empty results on page 66.


## 3. Inspect fields and data quality

This is an initial audit, not full cleaning. The notebook shows the fields returned and checks common identifier, date, latitude, and longitude names. Nepal bounds are a broad sanity check, not confirmation that a location is correct.

In [3]:
df = pd.DataFrame.from_records(records)
print(f'Record table shape: {df.shape}')
print('API fields:', sorted(map(str, df.columns)))
display(df.head(5))
def find_col(candidates):
    return next((name for name in candidates if name in df.columns), None)
id_col = find_col(['id','incident_id','uuid'])
date_col = find_col(['incident_on','created_on','date','incident_date','created_at'])
lat_col = find_col(['latitude','lat'])
lon_col = find_col(['longitude','long','lon','lng'])
audit = {'records_received':len(df), 'fields_received':len(df.columns), 'api_reported_total':reported_total, 'pagination_complete':retrieval_complete, 'stop_reason':stop_reason}
if id_col:
    audit.update({'identifier_field':id_col, 'missing_identifiers':int(df[id_col].isna().sum()), 'duplicate_identifiers':int(df[id_col].duplicated().sum())})
else: audit['identifier_field'] = 'not found among common names'
if date_col:
    dates = pd.to_datetime(df[date_col], errors='coerce', utc=True)
    audit.update({'date_field':date_col, 'unparseable_or_missing_dates':int(dates.isna().sum())})
    if dates.notna().any(): audit.update({'observed_date_min_utc':dates.min().isoformat(), 'observed_date_max_utc':dates.max().isoformat()})
else: audit['date_field'] = 'not found among common names'
if lat_col and lon_col:
    lat, lon = pd.to_numeric(df[lat_col], errors='coerce'), pd.to_numeric(df[lon_col], errors='coerce')
    inside = lat.between(26,31) & lon.between(80,89)
    audit.update({'latitude_field':lat_col, 'longitude_field':lon_col, 'missing_or_non_numeric_coordinates':int((lat.isna() | lon.isna()).sum()), 'outside_broad_nepal_bounds':int((~inside & lat.notna() & lon.notna()).sum())})
else: audit['coordinate_fields'] = 'not identified among common names'
display(pd.Series(audit, name='value').to_frame())

Record table shape: (64735, 28)
API fields: ['approved', 'cause', 'createdBy', 'createdOn', 'dataSource', 'dataSourceId', 'description', 'detail', 'event', 'hazard', 'id', 'incidentOn', 'loss', 'modifiedOn', 'needFollowup', 'point', 'polygon', 'region', 'regionId', 'reportedOn', 'source', 'streetAddress', 'title', 'titleNe', 'updatedBy', 'verificationMessage', 'verified', 'wards']


,id,title,wards,point,createdOn,modifiedOn,titleNe,description,cause,verified,...,region,regionId,dataSourceId,dataSource,source,event,hazard,loss,createdBy,updatedBy
0,64504,"Fire at Ram Gaau, Rapti Sonari Rural Municipal...",[2164],"{'type': 'Point', 'coordinates': [81.84765, 28...",2023-06-02T10:32:17.764099+05:45,2025-10-28T15:53:36.332145+05:45,"लुम्बिनी, बाँके, राप्तीसोनारी-२ मा आगलागी",NaN,NaN,True,...,NaN,NaN,60500.0,drr_api,other,NaN,10,211946,NaN,NaN
1,76453,Fire at Bhojpur Municipality-11,[317],"{'type': 'Point', 'coordinates': [87.08673681,...",2025-01-24T10:33:07.304099+05:45,2025-10-28T15:48:41.298032+05:45,"कोशी, भोजपुर, भोजपुर-११ मा आगलागी",NaN,NaN,True,...,NaN,NaN,72496.0,drr_api,nepal_police,NaN,10,341637,NaN,NaN
2,46998,"Fire at Buspark, Itahari Submetropolitan City-9",[4746],"{'type': 'Point', 'coordinates': [87.27973357,...",2020-06-14T11:20:16.409538+05:45,2025-10-28T15:58:15.251979+05:45,NaN,NaN,NaN,True,...,NaN,NaN,46998.0,drr_api,nepal_police,NaN,10,46356,NaN,NaN
3,40743,Fire at Malarani Rural Municipality-3,[20],"{'type': 'Point', 'coordinates': [83.12607602,...",2020-02-28T05:07:16.105515+05:45,2025-10-28T15:52:43.684913+05:45,NaN,NaN,NaN,True,...,NaN,NaN,40743.0,drr_api,nepal_police,NaN,10,29145,NaN,NaN
4,46106,"Fire at Pipariya, Bhimdatta Municipality-13",[6258],"{'type': 'Point', 'coordinates': [80.12227449,...",2020-03-19T11:56:20.698296+05:45,2025-10-28T15:58:32.283810+05:45,NaN,NaN,NaN,True,...,NaN,NaN,46106.0,drr_api,nepal_police,NaN,10,432407,NaN,NaN


,value
records_received,64735
fields_received,28
api_reported_total,9223372036854775807
pagination_complete,False
stop_reason,Empty results on page 66.
identifier_field,id
missing_identifiers,0
duplicate_identifiers,0
date_field,not found among common names
coordinate_fields,not identified among common names


## 4. Save records and the retrieval manifest

Raw page JSON is the preserved source snapshot. `records.json` and `records.csv` are convenience files for inspection. Keep the manifest and notebook outputs for the next analysis step; coverage claims will depend on the observed pagination and record dates.

In [4]:
records_json = run_dir / 'records.json'
records_json.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')
records_csv = run_dir / 'records.csv'
df.to_csv(records_csv, index=False, encoding='utf-8-sig')
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda:handle.read(1024*1024), b''): digest.update(block)
    return digest.hexdigest()
manifest = {'project':'UrbanPulse Nepal', 'source':'BIPAD Portal API', 'documentation_url':'https://bipadportal.gov.np/api/', 'endpoint':INCIDENT_ENDPOINT, 'retrieved_at_utc':retrieved_at.isoformat(), 'request_timeout_seconds':REQUEST_TIMEOUT_SECONDS, 'max_pages_guard':MAX_PAGES, 'pages_saved':len(pages), 'records_received':len(records), 'api_reported_total':reported_total, 'pagination_complete':retrieval_complete, 'stop_reason':stop_reason, 'page_errors':page_errors, 'pages':pages, 'sha256':{records_json.name:sha256(records_json), records_csv.name:sha256(records_csv)}}
manifest_path = run_dir / 'retrieval_manifest.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Output folder: {run_dir}')
print(f'Manifest: {manifest_path}')
print(f'CSV: {records_csv}')
print('Please share the final notebook outputs and retrieval_manifest.json for the next step.')

Output folder: c:\Projects_Tracking\DisasterLens Nepal\data\raw\bipad_incidents\20261013T101245Z
Manifest: c:\Projects_Tracking\DisasterLens Nepal\data\raw\bipad_incidents\20261013T101245Z\retrieval_manifest.json
CSV: c:\Projects_Tracking\DisasterLens Nepal\data\raw\bipad_incidents\20261013T101245Z\records.csv
Please share the final notebook outputs and retrieval_manifest.json for the next step.
